## Morphology locations as a Form

This is probably not very useful. Morphology locations are a rather abstract concept.
The form simply plots the generate locations on the morphology and writes them to a csv file.

In [ ]:
from obi_one.core.run_tasks import run_tasks_for_generated_scan
from obi_one.core.scan_generation import GridScanGenerationTask

from obi_one.scientific.blocks.morphology_locations.clustered import (
    ClusteredMorphologyLocations,
    ClusteredPathDistanceMorphologyLocations,
)
from obi_one.scientific.blocks.morphology_locations.explicit import (
    ExplicitMorphologyLocations,
    MorphologyLocationPoint,
)
from obi_one.scientific.blocks.morphology_locations.random import (
    RandomGroupedMorphologyLocations,
    RandomMorphologyLocations,
)
from obi_one.scientific.tasks.morphology_locations import MorphologyLocationsScanConfig
from pathlib import Path

morph_path = Path("../../../../data/morphologies/L5_TPC:A_MusMusculus_PrimarySomatosensory_C060114A5/C060114A5.swc")

In [ ]:
morph_locations = RandomMorphologyLocations(number_of_locations=[10, 20], section_types=(4, ))
# Try this one for a more exciting example.
# morph_locations = ClusteredPathDistanceMorphologyLocations(
#                         number_of_locations=[100, 200],
#                         section_types=(4, ),
#                         n_clusters=10,
#                         cluster_max_distance=20,
#                         path_dist_mean=200,
#                         path_dist_sd=80,
#                         n_groups_per_cluster=2,
#                     )
morph_loc_scan_config = MorphologyLocationsScanConfig(
                    initialize=MorphologyLocationsScanConfig.Initialize(
                        morphology=[morph_path]
                    ),
                    morph_locations=morph_locations
)

grid_scan = GridScanGenerationTask(form=morph_loc_scan_config,
                         output_root="../../../../../../obi-output/morphology_locations")
grid_scan.execute()
run_tasks_for_generated_scan(grid_scan)

## Morphology locations blocks

There are several blocks of differing complexity that generate locations. Above, we used the simplest one, but others are available.

Here, we demo most of them.

In [ ]:
from morphio import Morphology as MorphioMorphology, Option
m = MorphioMorphology(morph_path, options=Option.nrn_order)

### ExplicitMorphologyLocations

Use `ExplicitMorphologyLocations` for one or more deterministic targets. Each `MorphologyLocationPoint` contains a SONATA global section ID and normalized section offset. Section `0` is the soma; neurite section `N` corresponds to `m.section(N - 1)` when the morphology uses NEURON-compatible ordering.

In [ ]:
single_location = ExplicitMorphologyLocations(
    locations=(MorphologyLocationPoint(section_id=1, offset=0.5),)
)

display(single_location.points_on(m))

#### Multiple locations

The same block represents multiple interactive selections by adding more points.

In [ ]:
explicit_locations = ExplicitMorphologyLocations(
    locations=(
        MorphologyLocationPoint(section_id=1, offset=0.25),
        MorphologyLocationPoint(section_id=2, offset=0.75),
    )
)

display(explicit_locations.points_on(m))

### RandomMorphologyLocations
Simplest case: Random locations without constraints.

Note that there are no constraints on path distance or anything, really.

In [ ]:
# Default number of locations is 1
tst = RandomMorphologyLocations(section_types=(3, 4))
display(tst.points_on(m))

# Can specify a larger number instead.
tst = RandomMorphologyLocations(number_of_locations=10, section_types=(3, 4))
display(tst.points_on(m))

# Can also limit the section types.
# Note that the above likely has different section types, this one only shows "4".
tst = RandomMorphologyLocations(number_of_locations=10, section_types=(4, ))
display(tst.points_on(m))

### RandomGroupedMorphologyLocations
This only adds a grouping to the output. That is, the column "source_index" can have values other than 0.

In [ ]:
# Default is a single group. This does not change anything.
tst = RandomGroupedMorphologyLocations(section_types=(3, 4))
display(tst.points_on(m))

# More customized. Now "source_index" has different values
tst = RandomGroupedMorphologyLocations(number_of_locations=10, n_groups=2, section_types=(3, 4))
display(tst.points_on(m))

### ClusteredMorphologyLocations
This constrains generated locations such that they are clustered. That means, groups of them will be within a specified maximum distance of each other. Note that this distance is two times the value of the parameter 'cluster_max_distance'.

In [ ]:
# Note that many locations will be on the same section and nearby segments.
tst = ClusteredMorphologyLocations(
    number_of_locations=20, n_clusters=5, cluster_max_distance=5.0, section_types=(3, 4)
)
display(tst.points_on(m))

# Section types can be restricted further, e.g. to basal dendrites only.
tst = ClusteredMorphologyLocations(number_of_locations=20, n_clusters=5,
                                       cluster_max_distance=5.0, section_types=(3,))
display(tst.points_on(m))

### ClusteredPathDistanceMorphologyLocations
This exposes the full complexity: Several clusters at specific path distances, and several groups per cluster.

In [ ]:
tst = ClusteredPathDistanceMorphologyLocations(
    number_of_locations=100,
    n_clusters=5,
    cluster_max_distance=10.0,
    path_dist_mean=100.0,
    path_dist_sd=2.0,
    section_types=(3, 4)
)
display(tst.points_on(m))
